# RAG Application Using Typesense

Keyword search over a books collection, then a LangChain RAG pipeline with Typesense as the vector store and Groq as the LLM.

Set `TYPESENSE_HOST`, `TYPESENSE_API_KEY`, and `GROQ_API_KEY` in a `.env` file (or your environment) before running.

In [16]:
import os

import typesense
from dotenv import load_dotenv

load_dotenv()

TYPESENSE_HOST = os.getenv("TYPESENSE_HOST", "8a5wnxyt2oh9p-1.a1.typesense.net")
TYPESENSE_API_KEY = os.getenv("TYPESENSE_API_KEY", "kfl7rI0OUu4LrE3ZiiZfhvS1elvM")

client = typesense.Client({
    "nodes": [{
        "host": TYPESENSE_HOST,
        "port": "443",
        "protocol": "https",
    }],
    "api_key": TYPESENSE_API_KEY,
    "connection_timeout_seconds": 10,
})

client

In [17]:
books_schema = {
    "name": "books",
    "fields": [
        {"name": "title", "type": "string"},
        {"name": "authors", "type": "string[]", "facet": True},
        {"name": "publication_year", "type": "int32", "facet": True},
        {"name": "ratings_count", "type": "int32"},
        {"name": "average_rating", "type": "float"},
    ],
    "default_sorting_field": "ratings_count",
}

if "books" in client.collections:
    print("Collection 'books' already exists")
else:
    print(client.collections.create(books_schema))

ConnectError: [Errno 11001] getaddrinfo failed

In [ ]:
client.collections.retrieve()

In [ ]:
with open("books.jsonl", "r", encoding="utf-8") as jsonl_file:
    data = jsonl_file.read().strip()

if not data:
    raise ValueError("books.jsonl is empty. Add one JSON object per line before importing.")

import_result = client.collections["books"].documents.import_(data, {"action": "upsert"})
print(import_result)

In [ ]:
search_parameters = {
    "q": "harry potter",
    "query_by": "title,authors",
    "sort_by": "ratings_count:desc",
}

client.collections["books"].documents.search(search_parameters)

In [ ]:
search_parameters = {
    "q": "harry potter",
    "query_by": "title",
    "filter_by": "publication_year:<1998",
    "sort_by": "publication_year:desc",
}

client.collections["books"].documents.search(search_parameters)

In [ ]:
search_parameters = {
    "q": "experyment",
    "query_by": "title",
    "facet_by": "authors",
    "sort_by": "average_rating:desc",
}

client.collections["books"].documents.search(search_parameters)

In [ ]:
# LangChain + Typesense + Groq LLM RAG

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


class SentenceTransformerEmbeddings(Embeddings):
    """Local embeddings compatible with LangChain 1.x (HuggingFaceEmbeddings moved out of langchain.embeddings)."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.encode(texts, show_progress_bar=True).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.model.encode(text).tolist()

In [ ]:
print("Typesense host:", TYPESENSE_HOST)
print("GROQ_API_KEY is set:", bool(os.getenv("GROQ_API_KEY")))

In [ ]:
from pathlib import Path

text_dir = Path("data/text_files")
documents = []
for path in sorted(text_dir.glob("*.txt")):
    documents.extend(TextLoader(str(path), encoding="utf-8").load())

if not documents:
    raise FileNotFoundError(f"No .txt files found in {text_dir.resolve()}")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)
print(f"Loaded {len(documents)} documents, split into {len(docs)} chunks")

embeddings = SentenceTransformerEmbeddings()

In [ ]:
COLLECTION_NAME = "lang-chain"

if COLLECTION_NAME in client.collections:
    client.collections[COLLECTION_NAME].delete()

# typesense_collection_name must live inside typesense_client_params.
# LangChain's from_texts() swallows that argument and never passes it to the constructor.
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": TYPESENSE_HOST,
        "port": "443",
        "protocol": "https",
        "typesense_api_key": TYPESENSE_API_KEY,
        "typesense_collection_name": COLLECTION_NAME,
    },
)
docsearch

In [ ]:
query = "What is artificial intelligence"
found_docs = docsearch.similarity_search(query, k=3)
print(found_docs[0].page_content)

In [ ]:
retriever = docsearch.as_retriever(search_kwargs={"k": 3})
retriever

In [ ]:
query = "Artificial intelligence in-depth explanation"
retriever.invoke(query)[0]

In [ ]:
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("Set GROQ_API_KEY in your environment or a .env file before running this cell.")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.
If the context does not contain the answer, say you do not know.

Context:
{context}

Question: {question}
"""
)


def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is artificial intelligence?"))